In [23]:
# =====================================
# IMPORTS
# =====================================
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

# =====================================
# REPRODUCIBILITY
# =====================================
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# =====================================
# LOAD DATA
# =====================================
X = np.load("models/X.npy")
y = np.load("models/y.npy")

print("Dataset loaded")
print("Original X shape:", X.shape)

# =====================================
# NORMALIZATION (CRITICAL FIX)
# =====================================
mean = np.mean(X)
std = np.std(X) + 1e-8

X = (X - mean) / std

# SAVE normalization parameters
np.save("models/feature_mean.npy", mean)
np.save("models/feature_std.npy", std)

print("✅ Feature normalization saved")

# =====================================
# ADD CHANNEL DIMENSION
# =====================================
X = X[..., np.newaxis]
print("New X shape:", X.shape)

# =====================================
# TRAIN TEST SPLIT
# =====================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# MODEL ARCHITECTURE
# =====================================
model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(40, 64, 1)),

    tf.keras.layers.Conv2D(16, (3,3), activation='relu', padding="same"),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding="same"),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(len(np.unique(y)), activation='softmax')
])

# =====================================
# COMPILE
# =====================================
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0007)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =====================================
# CALLBACKS
# =====================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "models/best_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# =====================================
# TRAIN
# =====================================
history = model.fit(
    X_train,
    y_train,
    epochs=60,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, checkpoint]
)

# =====================================
# SAVE FINAL MODEL
# =====================================
os.makedirs("models", exist_ok=True)
model.save("models/anu_assistant_model.h5")

print("\n✅ Training complete")
print("✅ BEST model saved as models/best_model.h5")


Dataset loaded
Original X shape: (740, 40, 64)
✅ Feature normalization saved
New X shape: (740, 40, 64, 1)


Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_38 (Conv2D)              │ (None, 40, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_38 (MaxPooling2D) │ (None, 20, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 20, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 10, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_19 (Flatten)            │ (None, 5120)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 64)             │       327,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,324 (1.27 MB)

 Trainable params: 333,324 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

... (9 outputs left out)


Epoch 10: finished saving model to models/best_model.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6199 - loss: 1.1025 - val_accuracy: 0.7297 - val_loss: 1.0356
Epoch 11/60
34/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6443 - loss: 1.0282
Epoch 11: val_accuracy did not improve from 0.72973
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6706 - loss: 1.0051 - val_accuracy: 0.7095 - val_loss: 0.9864
Epoch 12/60
33/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7115 - loss: 0.9200
Epoch 12: val_accuracy did not improve from 0.72973
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6993 - loss: 0.8965 - val_accuracy: 0.7095 - val_loss: 0.8848
Epoch 13/60
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7266 - loss: 0.8522
Epoch 13: val_accuracy did not improve from 0.72973
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7230 - loss: 0.8336 - val_accuracy: 0.7297 - val_loss: 0.8688
Epoch 14/60
35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0


Epoch 18: finished saving model to models/best_model.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7618 - loss: 0.6908 - val_accuracy: 0.7365 - val_loss: 0.7730
Epoch 19/60
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8020 - loss: 0.5738
Epoch 19: val_accuracy improved from 0.73649 to 0.76351, saving model to models/best_model.h5



Epoch 19: finished saving model to models/best_model.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7922 - loss: 0.5818 - val_accuracy: 0.7635 - val_loss: 0.7246
Epoch 20/60
35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7799 - loss: 0.6280
Epoch 20: val_accuracy did not improve from 0.76351
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7736 - loss: 0.6521 - val_accuracy: 0.7432 - val_loss: 0.7937
Epoch 21/60
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7724 - loss: 0.6181
Epoch 21: val_accuracy did not improve from 0.76351
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7872 - loss: 0.5816 - val_accuracy: 0.7500 - val_loss: 0.7548
Epoch 22/60
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8269 - loss: 0.4679
Epoch 22: val_accuracy did not improve from 0.76351
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8125 - loss: 0.5156 - val_accuracy: 0.7500 - val_loss: 0.8037
Epoch 23/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.


✅ Training complete
✅ BEST model saved as models/best_model.h5


In [24]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import pickle
import os

# ---------------- PATHS ----------------
if '__file__' in globals():
    BASE_DIR = os.path.dirname(__file__)
else:
    # __file__ is not defined in interactive environments (e.g. Jupyter)
    BASE_DIR = os.getcwd()


MODEL_PATH = os.path.join(BASE_DIR, "models", "best_model.h5")
ENCODER_PATH = os.path.join(BASE_DIR, "models", "label_encoder.pkl")

# =====================================
# LOAD DATA
# =====================================
X = np.load("models/X.npy")
y = np.load("models/y.npy")

# LOAD TRAINING NORMALIZATION (CRITICAL FIX)
mean = np.load("models/feature_mean.npy")
std = np.load("models/feature_std.npy")

X = (X - mean) / std

# add channel dimension
X = X[..., np.newaxis]

# SAME SPLIT AS TRAINING
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# LOAD MODEL
# =====================================
model = tf.keras.models.load_model(MODEL_PATH)

with open(ENCODER_PATH, "rb") as f:
    encoder = pickle.load(f)

print("✅ Model loaded")

# =====================================
# EVALUATE
# =====================================
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\n===== FINAL TEST RESULTS =====")
print(f"Test Accuracy : {accuracy*100:.2f}%")
print(f"Test Loss     : {loss:.4f}")

# =====================================
# PREDICTIONS
# =====================================
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))


✅ Model loaded

===== FINAL TEST RESULTS =====
Test Accuracy : 76.35%
Test Loss     : 0.7246

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

         anu       0.82      0.90      0.86        10
       close       0.64      0.73      0.68        22
        help       0.83      0.91      0.87        11
        mute       1.00      0.90      0.95        10
        next       0.73      0.73      0.73        11
        open       0.71      0.57      0.63        21
       pause       0.82      0.90      0.86        10
        play       0.82      0.82      0.82        11
    previous       0.78      0.64      0.70        11
  screenshot       1.00      0.82      0.90        11
       sleep       0.73      0.80      0.76        10
      unmute       0.58      0.70      0.64        10

    accuracy                           0.76       148
   macro avg       0.79      0.78      0.78       148
weighted avg       0.77      0.76      0.76       148


====